# 실습예제 2-4 · 불용어·정규화: 같은 뜻은 하나로, 의미 없는 말은 빼고

실습 데이터의 **빈도 상위 단어**를 보며 반복해서 다듬는 4가지 실제 규칙을 적용합니다.

### 목차
0. 환경 설정 & 2-3의 토큰화 함수 준비
1. 출발점: 빈도 상위 단어 확인
2. 규칙 01 · 일반 불용어 – 공통 불용어 목록에서 제거
3. 규칙 02 · 도메인 불용어 – STOPWORDS 추가 + `max_df=0.5`로 이중 차단
4. 규칙 03 · 표기 통일 – 동의어 사전으로 치환 (재미있다 → 재밌다)
5. 규칙 04 · 신조어 분리 – 사용자 사전 등록 `add_user_word("꿀잼")`
6. 전체 파이프라인으로 묶기 & 전/후 비교
7. 정리


## 0. 환경 설정 & 토큰화 함수 준비

In [ ]:
# 한국어 형태소 분석기 Kiwi 설치 (scikit-learn, pandas는 Colab 기본 제공)
!pip install -q kiwipiepy

In [ ]:
from collections import Counter
import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer

kiwi = Kiwi()   # ※ 규칙 04에서 사용자 사전을 추가할 분석기

# 실습 데이터: 영화 리뷰
reviews = [
    "영화가 재밌다. 영화는 길었지만 영화를 또 보고 싶다",
    "배우들의 연기가 정말 좋았고 스토리도 탄탄한 영화였다",
    "이 영화 진짜 꿀잼! 가성비 좋은 꿀잼 영화 추천합니다",
    "스토리가 너무 지루하다. 평점을 높게 준 이유를 모르겠다",
    "재미있다는 평이 많아서 봤는데 생각보다 재미있었다",
    "연출은 좋았지만 결말이 아쉬운 영화. 평점 7점",
    "배우 연기는 최고였는데 스토리는 조금 지루했다",
    "재밌어요 재밌어요! 가족과 함께 보기 좋은 영화",
    "음악과 영상미가 좋아서 극장에서 볼 만한 영화였다",
    "기대를 많이 했는데 노잼이었다. 평점이 아깝다",
    "주연 배우의 연기 덕분에 끝까지 몰입해서 본 영화",
    "이런 것 정도는 누구나 만들 수 있다. 시간이 아깝다",
]

In [ ]:
def tokenize_filtered(sentence, analyzer=None):
    """[2-3 복습] 형태소 분석 + 품사 필터
    - 명사(NNG·NNP): 2글자 이상
    - 동사·형용사(VV·VA): 어간 + '다'
    - 어근(XR) + 하(XSA/XSV): '지루하다'처럼 합쳐서 복원
    - 조사·어미·접사·기호 등은 버림
    """
    analyzer = analyzer or kiwi
    tokens = analyzer.tokenize(sentence)
    words = []
    for i, t in enumerate(tokens):
        if t.tag in ("NNG", "NNP") and len(t.form) >= 2:
            words.append(t.form)
        elif t.tag in ("VV", "VA"):
            words.append(t.form + "다")
        elif t.tag == "XR":
            nxt = tokens[i + 1] if i + 1 < len(tokens) else None
            if nxt is not None and nxt.tag in ("XSA", "XSV"):
                words.append(t.form + nxt.form + "다")
    return words

def top_words(docs_tokens, n=15):
    """토큰 리스트들의 리스트를 받아 빈도 상위 n개 반환"""
    return Counter(w for doc in docs_tokens for w in doc).most_common(n)

## 1. 출발점: 빈도 상위 단어 확인
규칙을 정할 때는 항상 **빈도 상위 단어**부터 봅니다. 여기서 이상한 단어를 찾아 하나씩 규칙을 추가합니다.

In [ ]:
tokens_v0 = [tokenize_filtered(r) for r in reviews]
print("[처리 전] 빈도 상위 15")
for w, c in top_words(tokens_v0):
    print(f"  {w:<8} {c}")

## 2. 규칙 01 · 일반 불용어
**문제**: `것 · 수 · 정도 · 하다 · 있다`처럼 어디에나 나오는 말 → 의미 없음
**대응**: 공통 불용어 목록에서 제거


In [ ]:
# ① 일반 불용어: 어떤 문서에서든 흔하게 등장하는 말
#    (명사 2글자 필터로 '것','수'는 이미 빠지지만, 목록에 넣어 두면 필터를 바꿔도 안전)
GENERAL_STOPWORDS = {
    "것", "수", "정도", "하다", "있다", "없다", "되다", "이다", "같다",
    "보다", "싶다", "주다", "누구", "이유", "생각", "때문",
}

def remove_stopwords(tokens, stopwords):
    """토큰 리스트에서 불용어를 제거"""
    return [w for w in tokens if w not in stopwords]

tokens_v1 = [remove_stopwords(t, GENERAL_STOPWORDS) for t in tokens_v0]
print("[규칙 01 적용 후] 빈도 상위 15")
print(top_words(tokens_v1))

## 3. 규칙 02 · 도메인 불용어
**문제**: 영화 리뷰엔 `영화 · 평점`이 거의 모든 문서에 등장 → 문서를 구분하는 힘(구분력)이 없음
**대응**: ① STOPWORDS에 직접 추가 + ② `CountVectorizer(max_df=0.5)`로 **절반 넘는 문서에 나오는 단어 자동 차단** (이중 차단)


In [ ]:
# ② 도메인 불용어: 이 데이터(영화 리뷰)에서만 의미 없는 단어
DOMAIN_STOPWORDS = {"영화", "평점"}
STOPWORDS = GENERAL_STOPWORDS | DOMAIN_STOPWORDS   # 두 목록 합치기

tokens_v2 = [remove_stopwords(t, STOPWORDS) for t in tokens_v0]
print("[규칙 02 - STOPWORDS 추가 후] 빈도 상위 15")
print(top_words(tokens_v2))

In [ ]:
# 문서 빈도(DF) 확인: 각 단어가 '몇 개의 리뷰'에 등장했는지 비율
n_docs = len(tokens_v0)
df_ratio = Counter(w for doc in tokens_v0 for w in set(doc))   # set: 문서당 1번만 셈
print("문서 비율 상위 5:")
for w, c in df_ratio.most_common(5):
    print(f"  {w:<6} {c}/{n_docs}개 문서 ({c / n_docs:.0%})")

In [ ]:
# max_df로 자동 차단: 전체 문서의 50% '초과'로 등장하는 단어는 어휘에서 제외
#  - tokenizer : 우리가 만든 형태소 토크나이저를 그대로 사용
#  - stop_words: 직접 만든 불용어 목록
#  - max_df=0.5: 문서 비율 50% 초과 단어 자동 제거 (목록에 빠뜨린 도메인 불용어 대비)
#  - token_pattern=None: tokenizer를 직접 지정할 때 경고 방지
vectorizer = CountVectorizer(
    tokenizer=tokenize_filtered,
    lowercase=False,
    stop_words=list(STOPWORDS),
    max_df=0.5,
    token_pattern=None,
)

# ※ 시연용: 불용어 목록 없이 max_df만으로도 '영화'가 걸러지는지 확인
only_maxdf = CountVectorizer(tokenizer=tokenize_filtered, lowercase=False,
                             max_df=0.5, token_pattern=None).fit(reviews)
all_words = {w for doc in tokens_v0 for w in doc}              # 전체 단어
blocked = all_words - set(only_maxdf.vocabulary_)              # 어휘에서 빠진 단어
print("max_df=0.5만으로 차단된 단어:", sorted(blocked))        # → ['영화']

X = vectorizer.fit_transform(reviews)
print("최종 어휘 수:", len(vectorizer.vocabulary_))
print("어휘 일부  :", sorted(vectorizer.vocabulary_)[:15])

## 4. 규칙 03 · 표기 통일
**문제**: `재밌다 · 재미있다 · 재밌`이 따로 집계됨 (같은 뜻, 다른 표기)
**대응**: **동의어 사전**으로 대표 표기 하나로 치환 → `재미있다 → 재밌다`


In [ ]:
# 표기 통일 전: '재미' 관련 단어가 흩어져 있음
print("통일 전:", {w: c for w, c in Counter(w for d in tokens_v2 for w in d).items() if "재" in w})

# ③ 동의어 사전: {다른 표기: 대표 표기}
SYNONYMS = {
    "재미있다": "재밌다",
    "재밋다": "재밌다",
    "노잼": "재미없다",
    "재미없다": "재미없다",
}

def normalize(tokens, synonyms):
    """사전에 있는 단어는 대표 표기로 바꾸고, 없으면 그대로 둠"""
    return [synonyms.get(w, w) for w in tokens]

tokens_v3 = [normalize(t, SYNONYMS) for t in tokens_v2]
print("통일 후:", {w: c for w, c in Counter(w for d in tokens_v3 for w in d).items() if "재" in w})

## 5. 규칙 04 · 신조어 분리
**문제**: `꿀잼`이 `꿀 + 잼`으로 쪼개짐 (분석기 사전에 없는 신조어)
**대응**: **사용자 사전 등록** `kiwi.add_user_word("꿀잼", "NNG")`


In [ ]:
sample = "가성비 좋은 꿀잼 영화"

# 등록 전: 형태소만 뽑아서 확인
before = [t.form for t in kiwi.tokenize(sample)]
print("등록 전", before)      # ['가성비', '좋', '은', '꿀', '잼', '영화']

# ④ 사용자 사전에 신조어 등록 (단어, 품사, 점수)
#    점수(score)가 높을수록 해당 단어로 분석될 가능성이 커짐
kiwi.add_user_word("꿀잼", "NNG", 0)
kiwi.add_user_word("노잼", "NNG", 0)

after = [t.form for t in kiwi.tokenize(sample)]
print("등록 후", after)       # ['가성비', '좋', '은', '꿀잼', '영화']

## 6. 전체 파이프라인으로 묶기 & 전/후 비교
`형태소 분석 → 품사 필터 → 불용어 제거 → 표기 통일` 순서로 한 함수에 묶습니다.

In [ ]:
def preprocess(sentence):
    """2-3 + 2-4 전처리 전체 파이프라인"""
    words = tokenize_filtered(sentence)          # 형태소 분석 + 품사 필터 (사용자 사전 반영)
    words = normalize(words, SYNONYMS)           # 규칙 03: 표기 통일
    words = remove_stopwords(words, STOPWORDS)   # 규칙 01·02: 불용어 제거
    return words

tokens_final = [preprocess(r) for r in reviews]

# 처리 전/후 빈도 상위 단어 비교
compare = pd.DataFrame({
    "처리 전": [f"{w} ({c})" for w, c in top_words(tokens_v0, 10)],
    "처리 후": [f"{w} ({c})" for w, c in top_words(tokens_final, 10)],
})
compare

In [ ]:
# 리뷰별 전/후 결과
pd.set_option("display.max_colwidth", None)
pd.DataFrame({"리뷰": reviews, "처리 전": tokens_v0, "처리 후": tokens_final})

In [ ]:
# 최종: 파이프라인을 CountVectorizer에 연결 (+ max_df로 이중 차단)
final_vec = CountVectorizer(tokenizer=preprocess, lowercase=False,
                            max_df=0.5, token_pattern=None)
X = final_vec.fit_transform(reviews)

# 단어별 전체 빈도 상위 10
freq = pd.Series(X.sum(axis=0).A1, index=final_vec.get_feature_names_out())
freq.sort_values(ascending=False).head(10)

## 7. 정리

| # | 규칙 | 문제 | 대응 |
|---|---|---|---|
| 01 | 일반 불용어 | 것·수·정도·하다·있다 | 공통 불용어 목록에서 제거 |
| 02 | 도메인 불용어 | 영화·평점이 거의 모든 문서에 등장 | STOPWORDS 추가 + `max_df=0.5` 이중 차단 |
| 03 | 표기 통일 | 재밌다·재미있다 따로 집계 | 동의어 사전: 재미있다 → 재밌다 |
| 04 | 신조어 분리 | 꿀잼 → 꿀 + 잼 | `add_user_word("꿀잼")` |

👉 전처리는 한 번에 끝나지 않습니다. **빈도 상위 단어 확인 → 규칙 추가 → 다시 확인**을 반복하세요.
